# E-Commerce Customer Churn & RFM Segmentation Pipeline
### **Author:** Rishik Singh Thakur (Data Analyst)
### **Objective:** Identify high-value and at-risk customer cohorts to reduce a 4.5% MoM customer attrition rate and optimize targeting.  
### **Tech Stack:** Python (Pandas, NumPy, Scikit-Learn, Matplotlib, Seaborn)

---

## 1. Project Background & Business Context
An e-commerce brand noticed a **4.5% month-over-month decrease** in active customers. This cohort attrition was leading to an estimated revenue leakage of **$85,000 in at-risk contracts**. 

Instead of running generic promotional campaigns, the objective of this project is to:
1. Aggregate and clean raw transactional data (150,000+ rows).
2. Engineer **Recency, Frequency, and Monetary (RFM)** features.
3. Apply **K-Means Clustering** to segment customers into actionable cohorts.
4. Develop targeted retention strategies for each cohort, aiming to reclaim at-risk revenue and improve overall retention by **11%**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import os

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries loaded successfully.")

## 2. Load the Dataset
We will load the synthetic raw customers and transactions dataset generated for this case study. 
If the datasets do not exist, we will execute a helper generation script.

In [ ]:
# Check if data is available, otherwise generate
if not os.path.exists('../data/raw_transactions.csv'):
    print("Raw data not found! Generating data...")
    import sys
    sys.path.append('..')
    from scripts.generate_synthetic_data import generate_data
    generate_data()

transactions = pd.read_csv('../data/raw_transactions.csv')
customers = pd.read_csv('../data/raw_customers.csv')

print(f"Transactions Shape: {transactions.shape}")
print(f"Customers Shape: {customers.shape}")

## 3. Data Cleaning and Inspection
We will replicate our SQL pipeline's logic by identifying duplicates, handling negative amounts, and checking for missing keys.

In [ ]:
# 1. Drop duplicates
initial_len = len(transactions)
transactions.drop_duplicates(inplace=True)
print(f"Dropped {initial_len - len(transactions)} duplicate rows.")

# 2. Remove negative price anomalies (e.g. invalid transaction data)
anomalous_prices = (transactions['amount'] <= 0).sum()
transactions = transactions[transactions['amount'] > 0]
print(f"Removed {anomalous_prices} anomalous negative or zero-dollar transaction records.")

# 3. Handle missing customer IDs
missing_cust = transactions['customer_id'].isna().sum()
transactions.dropna(subset=['customer_id'], inplace=True)
print(f"Removed {missing_cust} orphan transaction rows with missing customer_id.")

# Check data types
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'])
customers['join_date'] = pd.to_datetime(customers['join_date'])
transactions.info()

## 4. Feature Engineering: RFM Metrics
We calculate: 
- **Recency (R)**: Days between the customer's last purchase and the baseline date ('2026-01-01').
- **Frequency (F)**: Total number of orders placed.
- **Monetary (M)**: Total spend.

In [ ]:
analysis_date = pd.to_datetime('2026-01-01')

rfm = transactions.groupby('customer_id').agg({
    'transaction_date': lambda x: (analysis_date - x.max()).days,
    'transaction_id': 'count',
    'amount': 'sum'
}).reset_index()

rfm.columns = ['customer_id', 'Recency', 'Frequency', 'Monetary']
print("RFM Features engineered. Summary statistics:")
print(rfm.describe())

## 5. Exploratory Data Analysis (EDA)
Let's look at the distribution of our engineered RFM features.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(rfm['Recency'], kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Recency Distribution (Days)')

sns.histplot(rfm['Frequency'], kde=True, ax=axes[1], color='salmon')
axes[1].set_title('Frequency Distribution (Orders)')

# Apply log-scale on spend due to long-tail skewness
sns.histplot(rfm['Monetary'], kde=True, ax=axes[2], color='lightgreen', log_scale=True)
axes[2].set_title('Monetary Distribution (Total Spend - Log Scale)')

plt.tight_layout()
plt.show()

## 6. Preprocessing & Feature Scaling
K-Means clustering is highly sensitive to variance in scale. We will apply a logarithmic transform to reduce right-skewness, and then apply `StandardScaler` to achieve a mean of 0 and variance of 1.

In [ ]:
# Log-transform to handle skew
rfm_log = rfm[['Recency', 'Frequency', 'Monetary']].copy()
# Ensure all values are positive before logging
rfm_log['Recency'] = rfm_log['Recency'] + 1 
rfm_log = np.log(rfm_log)

# Scale features
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

rfm_scaled_df = pd.DataFrame(rfm_scaled, columns=['Recency', 'Frequency', 'Monetary'])
print("Scaled RFM sample rows:")
print(rfm_scaled_df.head())

## 7. Determine Optimal Clusters: Elbow Method
We calculate the Within-Cluster Sum of Squares (WCSS) for range K=1 to K=10.

In [ ]:
wcss = []
k_range = range(1, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(rfm_scaled_df)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(8, 4.5))
plt.plot(k_range, wcss, marker='o', linestyle='--', color='purple')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('WCSS (Inertia)')
plt.title('Elbow Method for Optimal K')
plt.show()

From the Elbow plot, the rate of variance explained levels off around **K = 4**. We choose 4 clusters for our final segmentation.

In [ ]:
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled_df)

# Check cluster sizes
print("Cluster Customer Counts:")
print(rfm['Cluster'].value_counts())

## 8. Customer Profile Profiling
Let's aggregate the real RFM metrics by cluster and name the cohorts based on their features.

In [ ]:
cluster_profiles = rfm.groupby('Cluster').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': 'mean',
    'customer_id': 'count'
}).rename(columns={'customer_id': 'Size'}).reset_index()

# Sort clusters by spend to name them logically
cluster_profiles = cluster_profiles.sort_values(by='Monetary', ascending=False)
print(cluster_profiles)

Based on the average Recency, Frequency, and Monetary scores, we name the clusters as follows:
1. **Champions** (Low Recency, High Frequency, High Spend)
2. **Loyal Customer Base** (Moderate Recency, Moderate-High Frequency, High Spend)
3. **Recent / New Signups** (Very Low Recency, Low Frequency, Low Spend)
4. **At Risk / Churned** (Very High Recency, Low Frequency, Low Spend)

In [ ]:
# Mapping cluster IDs to friendly business segment labels
# (Note: Cluster order might fluctuate depending on random seed initialization)
# We will dynamically sort and map them to match profiles:

cluster_order = cluster_profiles['Cluster'].tolist()
segment_names = {
    cluster_order[0]: 'Champions (Core Assets)',
    cluster_order[1]: 'Loyals (High Engagement)',
    cluster_order[2]: 'Recent Signups (Growth Potential)',
    cluster_order[3]: 'Lost / At Risk (Needs Activation)'
}

rfm['Segment'] = rfm['Cluster'].map(segment_names)
print(rfm[['customer_id', 'Recency', 'Frequency', 'Monetary', 'Segment']].head())

## 9. Customer Segmentation Visualizations

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=rfm, 
    x='Recency', 
    y='Monetary', 
    hue='Segment', 
    palette='Set2',
    alpha=0.7
)
plt.title('Customer Segments: Recency vs Monetary Value')
plt.yscale('log')
plt.xlabel('Recency (Days Since Last Order)')
plt.ylabel('Monetary Value (Spend in USD - Log Scale)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 10. Business Impact & Strategic Recommendations

Based on the analysis, we propose the following tailored retention strategies:

### 1. **Champions (Core Assets)**
- **Action:** Enroll in the exclusive VIP Tier with early access to new releases. Offer dedicated customer support channels.
- **Goal:** Cultivate brand advocacy and drive referral loops.

### 2. **Loyals (High Engagement)**
- **Action:** Introduce targeted cross-selling campaigns based on past purchase categories. Offer loyalty points accelerators.
- **Goal:** Increase Average Order Value (AOV) and shift them to Champions.

### 3. **Recent Signups (Growth Potential)**
- **Action:** Launch an automated onboarding email flow containing a "second purchase" incentive (e.g., 15% off coupon within 30 days).
- **Goal:** Accelerate the time-to-second-purchase and reinforce habits.

### 4. **Lost / At Risk (Needs Activation)**
- **Action:** Deploy win-back re-engagement offers using localized SMS or emails containing deep discounts (e.g., "We miss you! Take $25 off your next order").
- **Goal:** Retrieve at least 15% of this segment, recovering approximately **$12,750 of the $85,000 at-risk baseline** and driving the target **11% overall retention increase**.